In [0]:
# Databricks notebook source

"""
table_runner.py

Purpose
-------
This notebook is responsible for executing ALL layers
for a SINGLE table.

Example

Workflow
    |
    |-- table_id = dim_customer
    |
    V

Layer0
   |
Success ?
   |
Layer1
   |
Success ?
   |
Layer2

If ANY layer fails,
execution stops immediately.

This notebook DOES NOT contain any business logic.

Actual transformation logic remains inside

orchestrator.py

This notebook simply controls the execution order.

Advantages
----------
✔ Workflow has only ONE For Each task

✔ Supports unlimited tables

✔ Easy debugging

✔ Production ready

"""


# -------------------------------------------------------------------
# Widgets
#
# Workflow passes only table_id.
#
# Example
#
# table_id = dim_customer
#
# -------------------------------------------------------------------

dbutils.widgets.text("table_id", "")
dbutils.widgets.text("catalog", "uc_dev_snt_fdn")
dbutils.widgets.text("job_run_id", "")
dbutils.widgets.text("environment", "dev")
table_id = "DIM_PRODUCT"
# table_id = dbutils.widgets.get("table_id")
catalog = dbutils.widgets.get("catalog")
job_run_id = dbutils.widgets.get("job_run_id")
environment = dbutils.widgets.get("environment")


# -------------------------------------------------------------------
# Layers to execute.
#
# We keep them in a list so that
#
# today -> 0,1,2
#
# tomorrow
#
# if Layer3 comes
#
# simply add "3"
#
# No code changes required.
#
# -------------------------------------------------------------------

layers = ["0", "1", "2"]


# -------------------------------------------------------------------
# Execute every layer.
#
# The orchestrator itself will
#
# 1 Read metadata
#
# 2 Check layer_is_active
#
# 3 Execute transformation
#
# 4 Write target
#
# 5 Audit
#
# 6 Return SUCCESS / SKIPPED / FAILED
#
# -------------------------------------------------------------------

for layer in layers:

    print("=" * 70)
    print(f"Starting Layer {layer} for {table_id}")
    print("=" * 70)

    try:

        result = dbutils.notebook.run(
            "/Workspace/Shared/snowflake_databricks_poc/notebooks/orchestrator.py",
            timeout_seconds=0,
            arguments={
                "table_id": table_id,
                "layer": layer,
                "catalog": catalog,
                "job_run_id": job_run_id,
                "environment": environment
            }
        )

        print(result)

        #
        # If layer is inactive,
        # orchestrator exits with
        #
        # "SKIPPED: layer not active"
        #
        # Based on your requirement,
        #
        # Layer1 disabled
        #
        # =>
        #
        # Stop execution.
        #

        if result == "SKIPPED":
            print(f"Execution stopped because Layer {layer} is disabled.")
            dbutils.notebook.exit("SKIPPED")
        elif result != "SUCCESS":
            raise Exception(f"Unexpected return value from orchestrator: {result}")


    except Exception as e:

        #
        # No logging here.
        #
        # orchestrator already logged
        # the failure.
        #
        # Just stop execution.
        #

        raise e


print(f"Completed all layers for {table_id}")

dbutils.notebook.exit("SUCCESS")